# Phase 2 — Baselines (scaffold)

Goal: establish a leaderboard of simple, no-ML baselines and pick the best one to submit. Every fancier
model in Phase 3 needs to beat these.

**This is a scaffold.** §0–§2 are boilerplate (filled in). §3 onwards are stubs with contracts and
implementation hints — you fill them in. Work through them in order.

**Recap from EDA that drives the design:**
- 773 train / 3 test wells. Train CSVs include the **ground-truth `TVT`** column alongside `TVT_input`,
  so validation = predict where `TVT_input` is NaN, compare to `TVT`.
- Eval zone is always contiguous and at the tail (100%). The boundary between the last known row and the
  first eval row is the natural anchor for every baseline.
- Eval-zone fraction: median 74%, range 20–88%.
- Eval zone is truly horizontal (median 88° from vertical) — `Z` is meaningful and observed everywhere.
- |ΔTVT/ΔZ| ≈ 1.05 step-by-step (not 1.0) — horizon dips along the wellbore, so a constant `TVT − Z`
  offset is a strong but imperfect baseline.
- GR has ~25–30% NaN. Baselines here don't use GR, so we sidestep that.

**Cells:**
- §0 Setup *(provided)*
- §1 Sample-submission inspection *(provided)*
- §2 Cache train wells in memory *(provided)*
- §3 Validation harness *(stub — you implement)*
- §4 Baseline A: carry-forward *(stub)*
- §5 Baseline B: Z-shift *(stub)*
- §6 Baseline C: linear extrapolation of TVT(MD) *(stub)*
- §7 Comparison *(notes only)*
- §8 Submission *(notes + pseudo-code)*
- §9 Findings & Phase-3 hooks *(template to fill)*

## 0. Setup *(provided)*

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

ROOT = Path('..').resolve()
DATA = ROOT / 'data'
TRAIN_DIR = DATA / 'train'
TEST_DIR  = DATA / 'test'
CACHE_DIR = ROOT / 'cache';        CACHE_DIR.mkdir(exist_ok=True)
SUB_DIR   = ROOT / 'submissions';  SUB_DIR.mkdir(exist_ok=True)

# Schema constants (confirmed from notebook 01)
COL_MD, COL_X, COL_Y, COL_Z = 'MD', 'X', 'Y', 'Z'
COL_GR = 'GR'
COL_TVT_INPUT = 'TVT_input'  # masked input (NaN in eval zone)
COL_TVT       = 'TVT'        # ground truth (train only)

print(f'Project root: {ROOT}')
print(f'Cache dir:    {CACHE_DIR}')
print(f'Submissions:  {SUB_DIR}')

## 1. Sample submission — lock the output format *(provided)*

Run this once. Note the column names (`ID_COL`, `VAL_COL`) — you'll match them when writing your own
submission. The row order in `sample_submission.csv` is canonical: predict every row it lists, in that order.

In [ ]:
ss = pd.read_csv(DATA / 'sample_submission.csv')
print(f'shape: {ss.shape}')
print(f'columns: {list(ss.columns)}')
print(ss.head(8))

ID_COL, VAL_COL = ss.columns[0], ss.columns[1]

# Confirm the {wellname}_{row_index} convention
ss_split = ss[ID_COL].str.rsplit('_', n=1, expand=True)
ss_split.columns = ['well', 'row_idx_str']
ss_split['row_idx'] = pd.to_numeric(ss_split['row_idx_str'], errors='coerce').astype('Int64')
print(f'\nWells in sample_submission: {ss_split["well"].nunique()}')
print(ss_split.groupby('well').size())

## 2. Cache all train wells in memory *(provided)*

Loads every train horizontal-well CSV into one DataFrame with `well` and `row_idx` columns. Pickles the
result so re-runs are instant. First run takes ~30–60s.

**Important columns added by this loader:** `well` (string) and `row_idx` (int — the row's position
*within its own well's CSV*, needed later for the submission row id).

In [ ]:
TRAIN_CACHE = CACHE_DIR / 'train_wells.pkl'

def load_train_wells(force_reload: bool = False) -> pd.DataFrame:
    if TRAIN_CACHE.exists() and not force_reload:
        df = pd.read_pickle(TRAIN_CACHE)
        print(f'Loaded cache: {df.shape} from {TRAIN_CACHE.name}')
        return df
    print('Reading 773 train CSVs...')
    files = sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
    frames = []
    for f in files:
        well = f.name.replace('__horizontal_well.csv', '')
        d = pd.read_csv(f)
        d['well'] = well
        d['row_idx'] = np.arange(len(d), dtype=np.int32)
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    df.to_pickle(TRAIN_CACHE)
    print(f'Cached: {df.shape} -> {TRAIN_CACHE.name}')
    return df

train_df = load_train_wells()
print(f'\nMemory: ~{train_df.memory_usage(deep=True).sum() / 1e6:.0f} MB')
print(f'Wells:  {train_df["well"].nunique()}')
train_df.head()

## 3. Validation harness *(implement)*

**This is the most important cell of the notebook.** Every baseline (and every Phase-3 model) plugs into
this function. Get the contract right and the rest follows.

**The contract:** a `predict_fn(well_df) -> np.ndarray` takes one well's DataFrame and returns predictions
for *every* row (length = `len(well_df)`), known and eval. The harness extracts the eval rows and scores them.

**Why predictions for every row, not just eval rows?** Because some baselines/models will use known TVT_input
internally (e.g. carry-forward via `ffill`), and forcing them to return a full-length array keeps the API
uniform. The harness throws away predictions on the known portion.

**What to implement:**
1. Loop over wells (`train_df.groupby('well', sort=False)`).
2. For each well: `eval_mask = g['TVT_input'].isna().values`. Skip if no eval rows.
3. Call `predict_fn(g)`, slice with `eval_mask`, compare to `g.loc[eval_mask, 'TVT']`.
4. Per-well: store RMSE, n_eval, anything else you'd want to inspect (MAE, max-error).
5. Pooled: accumulate `SSE_total` and `N_total` separately. Pooled RMSE = `sqrt(SSE_total / N_total)`.

**Pooled vs mean-of-per-well — why pooled?** The leaderboard scores `sqrt(mean((pred - truth)^2))` over
*all* eval rows in *all* test wells, so a long well contributes more than a short one. Mean-of-per-well-RMSEs
weights every well equally — different metric, different ranking. Always report both, but optimize for pooled.

**Sanity check after you implement it:** call it with a trivial baseline that returns `np.zeros(len(g))`. It
should produce huge RMSEs but not crash, and the per-well DataFrame should have 773 rows.

In [ ]:
def evaluate_baseline(
    predict_fn,
    train_df: pd.DataFrame,
    label: str = '',
) -> tuple[pd.DataFrame, float]:
    """Score a baseline against simulated-eval-zone ground truth.

    Args:
        predict_fn: function(well_df) -> array of length len(well_df). Predictions for every row.
        train_df:   concatenated train DataFrame from load_train_wells().
        label:      printed alongside the headline RMSE (optional).

    Returns:
        per_well: DataFrame with one row per well, columns ['well', 'rmse', 'n_eval', ...].
        pooled:   scalar pooled RMSE = sqrt(SSE_total / N_total) across all eval rows.
    """
    # TODO: implement (~10-15 lines). Hints in the markdown above.
    raise NotImplementedError


# --- sanity check (uncomment after implementing) ---
# def _zero_pred(g): return np.zeros(len(g))
# res, pooled = evaluate_baseline(_zero_pred, train_df, label='zeros')
# print(f'wells scored: {len(res)} | pooled RMSE: {pooled:.1f}')

## 4. Baseline A — carry-forward *(implement)*

Predict `TVT_eval = TVT_boundary` for every eval row — i.e. the last known TVT, carried forward unchanged.
Pure floor: assumes the well stays at the same TVT for the entire eval zone, which is rarely true. We expect
this to be the worst of the three; it's the number every other baseline must beat to justify its complexity.

**One-liner hint:** pandas has a method that does exactly this — `.ffill()`. Apply to `TVT_input`, return as
an array. Watch out: if the *first* row of a well is NaN there's nothing to fill from (shouldn't happen given
our EDA but worth defending against).

In [ ]:
def predict_carry_forward(g: pd.DataFrame) -> np.ndarray:
    # TODO: 1-2 lines.
    raise NotImplementedError


# res_A, rmse_A = evaluate_baseline(predict_carry_forward, train_df, 'A. carry-forward')
# res_A.describe()

## 5. Baseline B — Z-shift projection *(implement)*

Anchor at the boundary, assume the geological reference horizon is **flat** from there onward:

```
offset      = TVT_boundary - Z_boundary       (computed at the last known row)
TVT_eval[i] = Z_eval[i] + offset              (for every eval row i)
```

Should beat carry-forward whenever the well changes elevation through the eval zone (which is always — the
lateral isn't perfectly horizontal). The 5% dip drift we found in EDA means this baseline will accumulate
error toward the end of the eval zone, but should still be a clear win over A.

**Implementation steps:**
1. Find the last known row index (last position where `TVT_input` is *not* NaN).
2. Compute `offset = TVT_input[last] - Z[last]`.
3. For eval rows, fill in `Z[eval] + offset`. Leave known rows alone (use the original `TVT_input` values).

**Edge cases to defend against:**
- A well with no known rows at all (shouldn't happen given EDA — every well has a known portion — but bail
  gracefully if it does).
- The eval zone is the *first* rows of the well (also shouldn't happen — eval is always at the tail).

In [ ]:
def predict_z_shift(g: pd.DataFrame) -> np.ndarray:
    # TODO: ~6-8 lines.
    raise NotImplementedError


# res_B, rmse_B = evaluate_baseline(predict_z_shift, train_df, 'B. Z-shift')
# res_B.describe()

## 6. Baseline C — linear extrapolation of TVT(MD) *(implement)*

Fit a line to the last K known rows of `TVT_input` against `MD`, extrapolate over the eval zone. Captures
the local trend without leaning on `Z`. K is a hyperparameter — small K = noisy local fit; large K = washes
out recent build-curve dynamics. Sweep a few values: try K in {20, 50, 100, 200, 500, 1000} and report which
wins on pooled RMSE.

**Implementation:**
- `np.polyfit(x, y, 1)` returns `(slope, intercept)`.
- Take the last K known indices (or fewer if the known portion is shorter than K).
- Apply to MD values at the eval rows.

**Why this is interesting separate from B:** Z-shift is anchored to the boundary's `Z` and assumes flat
horizon. Linear-MD captures the *local TVT trend*, which is closer to the true horizon dip — but ignores the
fact that we *observe* Z in the eval zone. Comparing them tells you whether observed Z (B) or extrapolated
trend (C) is more informative. A future baseline could combine them.

In [ ]:
def make_linear_extrap_md(K: int):
    """Returns a predict_fn that fits a line on the last K known TVT_input rows."""
    def predict(g: pd.DataFrame) -> np.ndarray:
        # TODO: ~8-10 lines. Use np.polyfit on (MD, TVT_input) over last K known rows.
        raise NotImplementedError
    return predict


# K_sweep = [20, 50, 100, 200, 500, 1000]
# K_results = {}
# for K in K_sweep:
#     res, rmse = evaluate_baseline(make_linear_extrap_md(K), train_df, f'C linear K={K}')
#     K_results[K] = (res, rmse)
# best_K = min(K_results, key=lambda k: K_results[k][1])
# res_C, rmse_C = K_results[best_K]

## 7. Comparison *(implement)*

Once you have RMSEs from A, B, and C, build a comparison view. Suggested artifacts:

1. **Summary table** with one row per baseline, columns: pooled RMSE, median per-well RMSE, mean per-well
   RMSE, p90, max. Round to 2 decimals.
2. **Per-well RMSE distribution** — overlapping histograms of `res_A['rmse']`, `res_B['rmse']`, `res_C['rmse']`.
   Use a log x-axis since per-well RMSEs span a few orders of magnitude (some wells will be near-zero, others
   in the hundreds).
3. **Head-to-head scatter (optional but informative)** — `rmse_A` vs `rmse_B` per well, with the y=x line.
   Points below the line are wells where B beats A. Lets you ask: does B *uniformly* win, or only on average?
4. **Error-vs-position plot (optional)** — for each baseline, bin eval rows by relative position within the
   eval zone (0 at the boundary, 1 at the well's tail), plot mean |error|. Tells you whether error grows
   linearly with distance-from-boundary, plateaus, or stays flat. Big tells about which model class will help.

Plotting code from notebook 01 is reusable — same matplotlib idioms.

In [ ]:
# TODO: comparison table and plots.

## 8. Generate submission *(implement)*

Pick the best baseline by pooled RMSE, predict on the 3 test wells, write a CSV that matches
`sample_submission.csv` exactly (same row order, same column names).

**Steps:**
1. Pick the winning `predict_fn` and a sensible name (e.g. `'B_z_shift'`).
2. For each test horizontal-well CSV: read it, add a `row_idx` column, call `predict_fn(g)`, identify eval
   rows (`TVT_input.isna()`), and store predictions keyed by `f'{well}_{i}'` in a dict.
3. Map that dict onto `ss[ID_COL]` to get a Series in the canonical row order.
4. Assert no NaN remains in the value column. If there are NaNs: your eval-row detection on test wells
   produced different rows than `sample_submission` expects — investigate before submitting.
5. Write to `SUB_DIR / f'02_{baseline_name}.csv'`. Don't include the index.

**Pseudo-code structure:**
```python
predictions: dict[str, float] = {}
for f in sorted(TEST_DIR.glob('*__horizontal_well.csv')):
    well = f.name.replace('__horizontal_well.csv', '')
    g = pd.read_csv(f)
    g['row_idx'] = np.arange(len(g))
    pred = best_fn(g)
    eval_mask = g[COL_TVT_INPUT].isna().values
    for i in np.where(eval_mask)[0]:
        predictions[f'{well}_{i}'] = float(pred[i])

out = ss.copy()
out[VAL_COL] = out[ID_COL].map(predictions)
assert out[VAL_COL].notna().all(), 'missing predictions for some sample_submission rows'
out.to_csv(SUB_DIR / f'02_{baseline_name}.csv', index=False)
```

In [ ]:
# TODO: submission generation.

## 9. Findings & Phase-3 hooks

*Fill in after running the baselines.*

- **Best baseline:** ___, pooled RMSE ___
- **Carry-forward (A):** pooled RMSE ___
- **Z-shift (B):** pooled RMSE ___; beats A on ___ / 773 wells
- **Linear MD K=___ (C):** pooled RMSE ___; beats A on ___ / 773 wells
- **Where does error concentrate?** (uniform across eval zone / grows with distance / spikes at tail / etc.)
- **Which wells fail badly under all three?** (record their well IDs — Phase 3 will need to handle them)

**Phase 3 directions, in roughly increasing complexity:**
1. **Z-shift with locally-fit dip.** B assumes flat horizon. Fit `dip = d(TVT - Z)/d(MD)` from the last
   K known rows, propagate so the offset evolves along MD instead of staying constant. Cheap, likely a
   clear win over plain B.
2. **Predict surface elevations from train-only supervision.** Train a regressor on `(X, Y, Z, MD, GR)`
   -> surface columns (`ANCC, ASTNU, ASTNL, EGFDU, EGFDL, BUDA`) using the train data, then derive TVT from
   the predicted surface position at each test row. Uses train-only surface info as auxiliary supervision.
3. **Per-well GR<->typewell correlation.** Normalize lateral and typewell GR per-well; slide the lateral-GR
   window over the typewell GR-vs-TVT curve; take the best-matching position as the TVT estimate. The
   classic geosteering move.
4. **Global ML model.** XGBoost/LightGBM on engineered features per row: distance-from-boundary, local GR
   stats, trajectory derivatives, surface predictions, etc. Cross-well validation via grouped k-fold.
5. **Sequence models.** LSTM/Transformer over each well, encoder over the typewell, attention over the
   GR-correlation match. Defer until 1-4 are exhausted.

**Reusable pieces from this notebook:**
- `evaluate_baseline(predict_fn, train_df)` is the unit of measurement — every Phase-3 model plugs in here.
- `train_df` cache means cross-well experiments don't pay the 30s reload cost.
- The submission block in §8 only changes its `predict_fn` — same template across notebooks.